# Create the Customer H2O Environment

Materialize and register an immutable OpenJDK 17 environment pinned to the exact H2O version recorded in the customer manifest.

**Source:** Adapted from this repository's H2O deployment notebook and the Azure ML environment examples.

In [ ]:
from pathlib import Path
import json
import os

from azure.ai.ml import MLClient
from azure.ai.ml.entities import Environment
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / ".env.example").is_file() and (candidate / "pipelines").is_dir():
        WORKSHOP_ROOT = candidate
        break
else:
    raise FileNotFoundError("Run this notebook from inside the workshop folder")
load_dotenv(WORKSHOP_ROOT / ".env", override=True)

model_value = Path(os.environ["H2O_CUSTOMER_MODEL_PATH"])
MODEL_PATH = model_value if model_value.is_absolute() else WORKSHOP_ROOT / model_value
BUNDLE_DIR = MODEL_PATH.resolve().parent
manifest = json.loads((BUNDLE_DIR / "model_manifest.json").read_text(encoding="utf-8"))
if manifest["h2o_version"] != os.environ["H2O_VERSION"]:
    raise RuntimeError("The customer manifest and workshop H2O versions differ")

ENVIRONMENT_NAME = os.environ["H2O_ENVIRONMENT_NAME"]
ENVIRONMENT_VERSION = os.environ["H2O_ENVIRONMENT_VERSION"]
if ENVIRONMENT_VERSION.lower() == "latest":
    raise ValueError("Use an immutable environment version, not 'latest'")
REGISTER = os.getenv("REGISTER_H2O_ENVIRONMENT", "false").lower() in {"1", "true", "yes"}

output_dir = WORKSHOP_ROOT / "outputs/customer_h2o_environment"
output_dir.mkdir(parents=True, exist_ok=True)
conda_path = output_dir / "conda.yaml"
conda_path.write_text(
    f"""name: customer-h2o-binary
channels:
  - conda-forge
dependencies:
  - python=3.12
  - openjdk=17
  - pip
  - pip:
      - azureml-inference-server-http==1.4.1
      - h2o=={manifest['h2o_version']}
      - numpy==1.26.4
      - pandas==2.2.3
      - mlflow==2.22.1
      - azureml-mlflow==1.60.0.post1
""",
    encoding="utf-8",
)

environment_definition = Environment(
    name=ENVIRONMENT_NAME,
    version=ENVIRONMENT_VERSION,
    image="mcr.microsoft.com/azureml/minimal-py312-inference:latest",
    conda_file=str(conda_path),
    description="Exact OpenJDK 17 and H2O runtime for customer online and offline scoring",
    tags={"workshop": "azureml-h2o", "h2o_version": manifest["h2o_version"]},
)
credential = AzureCliCredential(tenant_id=os.getenv("AZURE_TENANT_ID") or None)
ml_client = MLClient(credential, os.environ["AZURE_SUBSCRIPTION_ID"], os.environ["AZURE_RESOURCE_GROUP"], os.environ["AZUREML_WORKSPACE_NAME"])

if REGISTER:
    registered_environment = ml_client.environments.create_or_update(environment_definition)
    verified_environment = ml_client.environments.get(ENVIRONMENT_NAME, ENVIRONMENT_VERSION)
    assert verified_environment.tags["h2o_version"] == manifest["h2o_version"]
    print(f"Registered environment: {verified_environment.name}:{verified_environment.version}")
else:
    print(f"Prepared environment: {ENVIRONMENT_NAME}:{ENVIRONMENT_VERSION}")
    print("Registration disabled. Set REGISTER_H2O_ENVIRONMENT=true in workshop/.env.")

## Expected Result

The generated Conda definition pins OpenJDK 17 and the manifest's exact H2O version, and the immutable Azure ML environment is retrievable.

Next: `04_deploy_online_endpoint.ipynb`.